# Notebook 08 — Chapter 4 Publication Tables, Figures, and Results Exports

This notebook converts the frozen Notebook 07 results package into publication-oriented Chapter 4 materials. It does **not** retrain models or alter hypothesis decisions. It:

- verifies the Notebook 07 checksum manifest and integrity checks;
- creates numbered Chapter 4 tables in CSV, Markdown, LaTeX, and HTML formats;
- generates high-resolution figures from the frozen point estimates and bootstrap intervals;
- writes a results narrative that references the generated tables and figures;
- creates caption, provenance, configuration, and checksum manifests; and
- packages the exports into a ZIP archive.

**Interpretation boundary:** the aircraft observations and NF-UNSW-NB15-v3 flows were joined through a controlled temporal replay experiment. The outputs do not establish causal or site-coincident aircraft/cyber incidents.

In [ ]:
from __future__ import annotations

import hashlib
import html
import json
import math
import os
import platform
import re
import sys
import zipfile
from datetime import datetime, timezone
from numbers import Integral, Real
from pathlib import Path
from typing import Any, Iterable, Mapping

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

EXPORT_VERSION = "chapter4_exports_v2"
SOURCE_RESULTS_VERSION = "results_v2"
TOP_EXPLANATION_COMPONENTS = int(
    os.environ.get("CHAPTER4_TOP_EXPLANATION_COMPONENTS", "8")
)
FIGURE_DPI = int(os.environ.get("CHAPTER4_FIGURE_DPI", "300"))


def locate_project_root(start: Path) -> Path:
    required = [
        Path("data/processed/point_event_v2/results_v2/results_reference_v2.json"),
        Path("data/processed/point_event_v2/results_v2/model_comparison_v2.csv"),
        Path("data/processed/point_event_v2/results_v2/SHA256SUMS.txt"),
    ]
    resolved_start = start.resolve()
    for candidate in [resolved_start, *resolved_start.parents]:
        if all((candidate / relative).is_file() for relative in required):
            return candidate
    return resolved_start


PROJECT_ROOT = locate_project_root(Path.cwd())
BASE_DIR = PROJECT_ROOT / "data" / "processed" / "point_event_v2"
RESULTS_DIR = BASE_DIR / SOURCE_RESULTS_VERSION
OUTPUT_DIR = BASE_DIR / EXPORT_VERSION
TABLES_DIR = OUTPUT_DIR / "tables"
FIGURES_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_IN = RESULTS_DIR / "results_reference_v2.json"
BUILD_SUMMARY_IN = RESULTS_DIR / "results_build_summary_v2.json"
CONFIG_IN = RESULTS_DIR / "results_config_v2.json"
SOURCE_CHECKSUMS_IN = RESULTS_DIR / "SHA256SUMS.txt"
SOURCE_INTEGRITY_IN = RESULTS_DIR / "results_integrity_checks_v2.csv"
RQ1_MAE_IN = RESULTS_DIR / "rq1_timestamp_alignment_mae_v2.csv"
RQ1_BOOTSTRAP_IN = RESULTS_DIR / "rq1_timestamp_alignment_bootstrap_v2.csv"
WINDOW_SENSITIVITY_IN = RESULTS_DIR / "alignment_window_sensitivity_v2.csv"
MODEL_COMPARISON_IN = RESULTS_DIR / "model_comparison_v2.csv"
MODEL_BOOTSTRAP_IN = RESULTS_DIR / "model_auprc_bootstrap_v2.csv"
PAIRED_DIFFERENCES_IN = RESULTS_DIR / "paired_model_auprc_differences_v2.csv"
HYPOTHESES_IN = RESULTS_DIR / "consolidated_hypothesis_evaluation_v2.csv"
SUBGROUP_IN = RESULTS_DIR / "subgroup_performance_v2.csv"
EXPLAINABILITY_IN = RESULTS_DIR / "explainability_summary_v2.csv"

CAPTION_MANIFEST_OUT = OUTPUT_DIR / "chapter4_caption_manifest_v2.csv"
INTEGRITY_OUT = OUTPUT_DIR / "chapter4_integrity_checks_v2.csv"
NARRATIVE_OUT = OUTPUT_DIR / "chapter4_results_draft_v2.md"
HTML_REPORT_OUT = OUTPUT_DIR / "chapter4_results_report_v2.html"
REFERENCE_OUT = OUTPUT_DIR / "chapter4_reference_v2.json"
CONFIG_OUT = OUTPUT_DIR / "chapter4_config_v2.json"
BUILD_SUMMARY_OUT = OUTPUT_DIR / "chapter4_build_summary_v2.json"
DATA_DICTIONARY_OUT = OUTPUT_DIR / "chapter4_data_dictionary_v2.csv"
CHECKSUMS_OUT = OUTPUT_DIR / "SHA256SUMS.txt"
PACKAGE_OUT = BASE_DIR / f"{EXPORT_VERSION}.zip"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": FIGURE_DPI,
        "font.size": 10,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
    }
)

print("Project root:", PROJECT_ROOT)
print("Notebook 07 results:", RESULTS_DIR)
print("Chapter 4 exports:", OUTPUT_DIR)
print("Figure DPI:", FIGURE_DPI)

In [ ]:
def require_text(value: object, name: str) -> str:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{name} is missing.")
    if isinstance(value, bytes):
        text = value.decode("utf-8", errors="strict")
    else:
        text = str(value)
    text = text.strip()
    if not text or text.lower() in {"nan", "nat", "none", "<na>"}:
        raise ValueError(f"{name} is empty or missing.")
    return text


def require_float(value: object, name: str) -> float:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{name} is missing.")
    if isinstance(value, (Real, np.number)):
        converted = float(value)
    else:
        text = require_text(value, name)
        try:
            converted = float(text)
        except ValueError as exc:
            raise ValueError(f"{name} is not numeric: {value!r}") from exc
    if not math.isfinite(converted):
        raise ValueError(f"{name} is not finite: {value!r}")
    return converted


def optional_float(value: object) -> float | None:
    if value is None or value is pd.NA or value is pd.NaT:
        return None
    try:
        converted = float(str(value))
    except (TypeError, ValueError):
        return None
    return converted if math.isfinite(converted) else None


def require_int(value: object, name: str) -> int:
    if value is None or value is pd.NA or value is pd.NaT:
        raise ValueError(f"{name} is missing.")
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (Integral, np.integer)):
        return int(value)
    if isinstance(value, (Real, np.floating)):
        converted = float(value)
        if not math.isfinite(converted) or not converted.is_integer():
            raise ValueError(f"{name} is not an integer: {value!r}")
        return int(converted)
    text = require_text(value, name)
    try:
        converted = float(text)
    except ValueError as exc:
        raise ValueError(f"{name} is not integer-compatible: {value!r}") from exc
    if not math.isfinite(converted) or not converted.is_integer():
        raise ValueError(f"{name} is not an integer: {value!r}")
    return int(converted)


def require_bool(value: object, name: str) -> bool:
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if isinstance(value, (Integral, np.integer)):
        integer_value = int(value)
        if integer_value in {0, 1}:
            return bool(integer_value)
    text = require_text(value, name).lower()
    if text in {"true", "1", "yes", "supported", "pass"}:
        return True
    if text in {"false", "0", "no", "not supported", "fail"}:
        return False
    raise ValueError(f"{name} is not boolean-compatible: {value!r}")


def require_columns(frame: pd.DataFrame, columns: Iterable[str], table_name: str) -> None:
    missing = sorted(set(columns) - set(frame.columns))
    if missing:
        raise ValueError(f"{table_name} is missing required columns: {missing}")


def read_json_object(path: Path) -> dict[str, Any]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(payload, dict):
        raise ValueError(f"Expected a JSON object in {path}")
    return payload


def require_mapping(value: object, name: str) -> Mapping[str, Any]:
    if not isinstance(value, Mapping):
        raise ValueError(f"{name} must be a mapping.")
    normalized: dict[str, Any] = {}
    for key, item in value.items():
        normalized[require_text(key, f"{name} key")] = item
    return normalized


def nested_value(mapping: Mapping[str, Any], path: Iterable[str], name: str) -> object:
    current: object = mapping
    for key in path:
        current_mapping = require_mapping(current, name)
        if key not in current_mapping:
            raise KeyError(f"{name} is missing key {key!r}.")
        current = current_mapping[key]
    return current


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_checksum_manifest(path: Path) -> dict[str, str]:
    entries: dict[str, str] = {}
    for line_number, raw_line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
        line = raw_line.strip()
        if not line:
            continue
        parts = line.split(maxsplit=1)
        if len(parts) != 2:
            raise ValueError(f"Invalid checksum line {line_number} in {path}")
        digest = require_text(parts[0], f"checksum digest line {line_number}")
        relative_name = require_text(parts[1], f"checksum path line {line_number}")
        if not re.fullmatch(r"[0-9a-fA-F]{64}", digest):
            raise ValueError(f"Invalid SHA-256 digest on line {line_number} in {path}")
        entries[relative_name] = digest.lower()
    if not entries:
        raise ValueError(f"Checksum manifest is empty: {path}")
    return entries


def numeric_array(frame: pd.DataFrame, column: str, table_name: str) -> np.ndarray:
    require_columns(frame, [column], table_name)
    numeric = pd.to_numeric(frame[column], errors="coerce")
    array = numeric.to_numpy(dtype=float, copy=False)
    if not np.isfinite(array).all():
        bad_count = require_int((~np.isfinite(array)).sum(), f"invalid {column} count")
        raise ValueError(f"{table_name}.{column} contains {bad_count:,} non-finite values.")
    return array


def format_number(value: object, digits: int = 3, signed: bool = False) -> str:
    converted = optional_float(value)
    if converted is None:
        return "—"
    prefix = "+" if signed and converted > 0 else ""
    return f"{prefix}{converted:.{digits}f}"


def format_percent(value: object, digits: int = 1) -> str:
    converted = optional_float(value)
    if converted is None:
        return "—"
    return f"{100.0 * converted:.{digits}f}%"


def format_count(value: object) -> str:
    return f"{require_int(value, 'count'):,}"


def support_label(value: object) -> str:
    return "Supported" if require_bool(value, "support decision") else "Not supported"


def dataframe_to_markdown(frame: pd.DataFrame) -> str:
    columns = [require_text(column, "column name") for column in frame.columns]
    escaped_columns = [column.replace("|", "\\|") for column in columns]
    lines = [
        "| " + " | ".join(escaped_columns) + " |",
        "| " + " | ".join(["---"] * len(columns)) + " |",
    ]
    for row in frame.itertuples(index=False, name=None):
        values = [str(value).replace("|", "\\|").replace("\n", " ") for value in row]
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines) + "\n"


def write_table_bundle(
    frame: pd.DataFrame,
    stem: str,
    caption: str,
    latex_label: str,
) -> list[Path]:
    csv_path = TABLES_DIR / f"{stem}.csv"
    markdown_path = TABLES_DIR / f"{stem}.md"
    latex_path = TABLES_DIR / f"{stem}.tex"
    html_path = TABLES_DIR / f"{stem}.html"

    frame.to_csv(csv_path, index=False)
    markdown_path.write_text(dataframe_to_markdown(frame), encoding="utf-8")
    latex_path.write_text(
        frame.to_latex(
            index=False,
            escape=True,
            caption=caption,
            label=latex_label,
        ),
        encoding="utf-8",
    )
    html_path.write_text(
        frame.to_html(index=False, border=0, justify="center"),
        encoding="utf-8",
    )
    return [csv_path, markdown_path, latex_path, html_path]


def row_where(frame: pd.DataFrame, column: str, value: str, table_name: str) -> pd.Series:
    require_columns(frame, [column], table_name)
    matched = frame.loc[frame[column].astype("string").eq(value)]
    if len(matched) != 1:
        raise ValueError(
            f"Expected exactly one {table_name} row where {column}={value!r}; found {len(matched)}."
        )
    return matched.iloc[0]


def slugify(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")

## Source validation

Notebook 08 treats Notebook 07 as frozen input. The following cell verifies every artifact listed in the Notebook 07 checksum manifest and confirms that the upstream integrity checks passed.

In [ ]:
required_files = [
    REFERENCE_IN,
    BUILD_SUMMARY_IN,
    CONFIG_IN,
    SOURCE_CHECKSUMS_IN,
    SOURCE_INTEGRITY_IN,
    RQ1_MAE_IN,
    RQ1_BOOTSTRAP_IN,
    WINDOW_SENSITIVITY_IN,
    MODEL_COMPARISON_IN,
    MODEL_BOOTSTRAP_IN,
    PAIRED_DIFFERENCES_IN,
    HYPOTHESES_IN,
    SUBGROUP_IN,
    EXPLAINABILITY_IN,
]
missing_files = [path for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        "Notebook 08 is missing required Notebook 07 artifacts:\n"
        + "\n".join(str(path) for path in missing_files)
    )

source_manifest = parse_checksum_manifest(SOURCE_CHECKSUMS_IN)
integrity_rows: list[dict[str, object]] = []

for relative_name, expected_digest in sorted(source_manifest.items()):
    source_path = RESULTS_DIR / relative_name
    exists = source_path.is_file()
    actual_digest = sha256_file(source_path) if exists else ""
    integrity_rows.append(
        {
            "check": f"source_checksum::{relative_name}",
            "expected": expected_digest,
            "actual": actual_digest if exists else "missing",
            "match": exists and actual_digest == expected_digest,
            "severity": "error",
        }
    )

source_integrity = pd.read_csv(SOURCE_INTEGRITY_IN)
require_columns(
    source_integrity,
    ["check", "expected", "actual", "match", "severity"],
    SOURCE_INTEGRITY_IN.name,
)
for record in source_integrity.to_dict(orient="records"):
    integrity_rows.append(
        {
            "check": f"notebook07::{require_text(record.get('check'), 'Notebook 07 check')}",
            "expected": record.get("expected"),
            "actual": record.get("actual"),
            "match": require_bool(record.get("match"), "Notebook 07 integrity match"),
            "severity": require_text(record.get("severity"), "Notebook 07 integrity severity"),
        }
    )

integrity_checks = pd.DataFrame(integrity_rows)
integrity_checks.to_csv(INTEGRITY_OUT, index=False)

failed_errors = integrity_checks.loc[
    integrity_checks["severity"].astype("string").str.lower().eq("error")
    & ~integrity_checks["match"].astype(bool)
]
if not failed_errors.empty:
    display(failed_errors)
    raise ValueError("Notebook 08 stopped because one or more source integrity checks failed.")

print(f"Verified {len(source_manifest):,} Notebook 07 checksums.")
print(f"Verified {len(source_integrity):,} Notebook 07 integrity checks.")

In [ ]:
results_reference = read_json_object(REFERENCE_IN)
results_build_summary = read_json_object(BUILD_SUMMARY_IN)
results_config = read_json_object(CONFIG_IN)

rq1_mae = pd.read_csv(RQ1_MAE_IN)
rq1_bootstrap = pd.read_csv(RQ1_BOOTSTRAP_IN)
window_sensitivity = pd.read_csv(WINDOW_SENSITIVITY_IN)
model_comparison = pd.read_csv(MODEL_COMPARISON_IN)
model_bootstrap = pd.read_csv(MODEL_BOOTSTRAP_IN)
paired_differences = pd.read_csv(PAIRED_DIFFERENCES_IN)
hypothesis_evaluation = pd.read_csv(HYPOTHESES_IN)
subgroup_performance = pd.read_csv(SUBGROUP_IN)
explainability = pd.read_csv(EXPLAINABILITY_IN)

require_columns(
    rq1_mae,
    [
        "subset",
        "rows",
        "event_time_mae_s",
        "processing_time_mae_s",
        "mae_reduction_s",
        "processing_time_median_absolute_error_s",
        "processing_time_p95_absolute_error_s",
        "processing_time_max_absolute_error_s",
        "h1_required_reduction_s",
        "h1_supported_point_estimate",
    ],
    RQ1_MAE_IN.name,
)
require_columns(
    rq1_bootstrap,
    [
        "comparison",
        "point_mae_reduction_s",
        "bootstrap_ci_lower_95_s",
        "bootstrap_ci_upper_95_s",
        "probability_reduction_at_least_1s",
        "bootstrap_iterations",
    ],
    RQ1_BOOTSTRAP_IN.name,
)
require_columns(
    model_comparison,
    [
        "model_key",
        "model_label",
        "model_family",
        "validation_auprc",
        "test_auprc",
        "test_roc_auc",
        "test_precision",
        "test_recall",
        "test_f1",
        "test_balanced_accuracy",
        "test_brier_score",
        "selected_threshold",
        "p95_latency_ms",
        "latency_measurement",
    ],
    MODEL_COMPARISON_IN.name,
)
require_columns(
    model_bootstrap,
    [
        "model_key",
        "point_test_auprc",
        "bootstrap_ci_lower_95",
        "bootstrap_ci_upper_95",
        "bootstrap_iterations",
        "model_label",
    ],
    MODEL_BOOTSTRAP_IN.name,
)
require_columns(
    paired_differences,
    [
        "comparison",
        "point_auprc_difference",
        "bootstrap_ci_lower_95",
        "bootstrap_ci_upper_95",
        "probability_difference_gt_zero",
        "required_margin",
        "probability_meets_required_margin",
        "point_margin_pass",
    ],
    PAIRED_DIFFERENCES_IN.name,
)
require_columns(
    hypothesis_evaluation,
    [
        "hypothesis",
        "research_question",
        "criterion",
        "observed_value",
        "required_value",
        "bootstrap_ci_lower_95",
        "bootstrap_ci_upper_95",
        "latency_value_ms",
        "latency_limit_ms",
        "supported",
    ],
    HYPOTHESES_IN.name,
)
require_columns(
    window_sensitivity,
    [
        "window_half_width_s",
        "window_name",
        "events",
        "events_with_flows",
        "flow_coverage_fraction",
        "positive_targets",
        "target_prevalence",
        "target_changes_vs_primary",
        "positive_jaccard_vs_primary",
        "boundary_truncated_events",
    ],
    WINDOW_SENSITIVITY_IN.name,
)
require_columns(
    subgroup_performance,
    [
        "model_key",
        "model_label",
        "subgroup_variable",
        "subgroup_value",
        "rows",
        "positives",
        "prevalence",
        "auprc",
        "f1",
    ],
    SUBGROUP_IN.name,
)
require_columns(
    explainability,
    [
        "model_family",
        "component_type",
        "component",
        "importance_metric",
        "importance",
        "rank_within_source",
        "variant",
    ],
    EXPLAINABILITY_IN.name,
)

print("Loaded frozen Notebook 07 result tables.")
display(model_comparison)

## Table 4.1 — Evaluation design and interpretation boundary

In [ ]:
input_summary = require_mapping(
    results_build_summary.get("inputs"),
    "Notebook 07 build summary inputs",
)
definition_summary = require_mapping(
    results_build_summary.get("definitions"),
    "Notebook 07 build summary definitions",
)
interpretation_summary = require_mapping(
    results_build_summary.get("interpretation"),
    "Notebook 07 build summary interpretation",
)

candidate_events = require_int(input_summary.get("event_rows"), "candidate event rows")
common_test_events = require_int(input_summary.get("common_test_events"), "common test events")
test_positives = require_int(input_summary.get("test_positives"), "test positives")
test_prevalence = require_float(input_summary.get("test_prevalence"), "test prevalence")
alignment_mode = require_text(input_summary.get("alignment_mode"), "alignment mode")
bootstrap_iterations = require_int(
    definition_summary.get("bootstrap_iterations"),
    "bootstrap iterations",
)
minimum_subgroup_rows = require_int(
    definition_summary.get("minimum_subgroup_rows"),
    "minimum subgroup rows",
)
primary_window_s = require_int(
    definition_summary.get("primary_window_half_width_s"),
    "primary window half-width",
)
interpretation_statement = require_text(
    interpretation_summary.get("statement"),
    "interpretation statement",
)

TABLE_4_1_CAPTION = (
    "Evaluation design, frozen sample counts, and interpretation boundary for the "
    "controlled cross-domain alignment experiment."
)

table_4_1 = pd.DataFrame(
    [
        {"Design element": "Candidate low-altitude point events", "Value": f"{candidate_events:,}"},
        {"Design element": "Common held-out events", "Value": f"{common_test_events:,}"},
        {"Design element": "Positive held-out events", "Value": f"{test_positives:,}"},
        {"Design element": "Held-out target prevalence", "Value": format_percent(test_prevalence, 2)},
        {"Design element": "Primary temporal window", "Value": f"±{primary_window_s:,} seconds"},
        {"Design element": "Alignment mode", "Value": alignment_mode.replace("_", " ")},
        {"Design element": "Paired bootstrap iterations", "Value": f"{bootstrap_iterations:,}"},
        {"Design element": "Minimum subgroup rows", "Value": f"{minimum_subgroup_rows:,}"},
        {"Design element": "Interpretation boundary", "Value": interpretation_statement},
    ]
)

table_4_1_paths = write_table_bundle(
    table_4_1,
    "table_4_1_evaluation_design_v2",
    TABLE_4_1_CAPTION,
    "tab:evaluation-design",
)
display(table_4_1)

## Table 4.2 and Figure 4.1 — RQ1/H1 timestamp fidelity

In [ ]:
h1_record = row_where(hypothesis_evaluation, "hypothesis", "H1", HYPOTHESES_IN.name)
h1_bootstrap_record = rq1_bootstrap.iloc[0]

TABLE_4_2_CAPTION = (
    "Event-time and processing-time timestamp fidelity by event subset. The all-events "
    "row carries the paired-bootstrap uncertainty interval used for H1."
)

table_4_2_rows: list[dict[str, object]] = []
for row_position in range(len(rq1_mae)):
    row = rq1_mae.iloc[row_position]
    subset = require_text(row["subset"], f"RQ1 subset row {row_position}")
    is_all_events = subset == "all_events"
    ci_text = "—"
    decision_text = "Descriptive"
    if is_all_events:
        ci_text = (
            f"[{format_number(h1_bootstrap_record['bootstrap_ci_lower_95_s'], 3)}, "
            f"{format_number(h1_bootstrap_record['bootstrap_ci_upper_95_s'], 3)}]"
        )
        decision_text = support_label(h1_record["supported"])
    table_4_2_rows.append(
        {
            "Subset": subset.replace("_", " ").title(),
            "n": format_count(row["rows"]),
            "Event-time MAE (s)": format_number(row["event_time_mae_s"], 3),
            "Processing-time MAE (s)": format_number(row["processing_time_mae_s"], 3),
            "MAE reduction (s)": format_number(row["mae_reduction_s"], 3),
            "Reduction 95% CI (s)": ci_text,
            "Processing-time median AE (s)": format_number(
                row["processing_time_median_absolute_error_s"], 3
            ),
            "Processing-time p95 AE (s)": format_number(
                row["processing_time_p95_absolute_error_s"], 3
            ),
            "H1 decision": decision_text,
        }
    )

table_4_2 = pd.DataFrame(table_4_2_rows)
table_4_2_paths = write_table_bundle(
    table_4_2,
    "table_4_2_timestamp_alignment_v2",
    TABLE_4_2_CAPTION,
    "tab:timestamp-alignment",
)
display(table_4_2)

all_events_row = row_where(rq1_mae, "subset", "all_events", RQ1_MAE_IN.name)
alignment_labels = ["Event time", "Processing time"]
alignment_values = np.asarray(
    [
        require_float(all_events_row["event_time_mae_s"], "event-time MAE"),
        require_float(all_events_row["processing_time_mae_s"], "processing-time MAE"),
    ],
    dtype=float,
)

figure_4_1_path = FIGURES_DIR / "figure_4_1_timestamp_alignment_mae.png"
plt.figure(figsize=(7.2, 4.8))
plt.bar(alignment_labels, alignment_values)
plt.ylabel("Mean absolute timestamp error (seconds)")
plt.title("Timestamp Fidelity: Event Time vs. Processing Time")
for position, value in enumerate(alignment_values.tolist()):
    plt.text(position, value, f"{value:.3f}", ha="center", va="bottom")
plt.tight_layout()
plt.savefig(figure_4_1_path, dpi=FIGURE_DPI)
plt.close()

## Table 4.3 and Figures 4.2–4.3 — Model performance and latency

In [ ]:
TABLE_4_3_CAPTION = (
    "Frozen validation and held-out performance for the selected fused tabular, temporal "
    "Transformer, full R-GCN, and R-GCN ablation models."
)

model_display = model_comparison.sort_values("test_auprc", ascending=False).reset_index(drop=True)
table_4_3_rows: list[dict[str, object]] = []
for row_position in range(len(model_display)):
    row = model_display.iloc[row_position]
    table_4_3_rows.append(
        {
            "Model": require_text(row["model_label"], f"model label row {row_position}"),
            "Validation AUPRC": format_number(row["validation_auprc"], 4),
            "Test AUPRC": format_number(row["test_auprc"], 4),
            "Test ROC-AUC": format_number(row["test_roc_auc"], 4),
            "Precision": format_number(row["test_precision"], 4),
            "Recall": format_number(row["test_recall"], 4),
            "F1": format_number(row["test_f1"], 4),
            "Balanced accuracy": format_number(row["test_balanced_accuracy"], 4),
            "Brier score": format_number(row["test_brier_score"], 4),
            "Threshold": format_number(row["selected_threshold"], 4),
            "p95 latency (ms)": format_number(row["p95_latency_ms"], 3),
        }
    )

table_4_3 = pd.DataFrame(table_4_3_rows)
table_4_3_paths = write_table_bundle(
    table_4_3,
    "table_4_3_model_performance_v2",
    TABLE_4_3_CAPTION,
    "tab:model-performance",
)
display(table_4_3)

bootstrap_plot = model_bootstrap.copy()
bootstrap_plot["point_test_auprc"] = pd.to_numeric(
    bootstrap_plot["point_test_auprc"], errors="raise"
)
bootstrap_plot = bootstrap_plot.sort_values("point_test_auprc", ascending=True).reset_index(drop=True)
point_values = numeric_array(bootstrap_plot, "point_test_auprc", MODEL_BOOTSTRAP_IN.name)
lower_values = numeric_array(bootstrap_plot, "bootstrap_ci_lower_95", MODEL_BOOTSTRAP_IN.name)
upper_values = numeric_array(bootstrap_plot, "bootstrap_ci_upper_95", MODEL_BOOTSTRAP_IN.name)
labels = bootstrap_plot["model_label"].astype("string").tolist()
positions = np.arange(len(bootstrap_plot), dtype=float)
left_errors = point_values - lower_values
right_errors = upper_values - point_values

figure_4_2_path = FIGURES_DIR / "figure_4_2_model_test_auprc_ci.png"
plt.figure(figsize=(9.0, 5.4))
plt.errorbar(
    point_values,
    positions,
    xerr=np.vstack([left_errors, right_errors]),
    fmt="o",
    capsize=4,
)
plt.yticks(positions, labels)
plt.xlabel("Held-out AUPRC")
plt.xlim(0.0, 1.02)
plt.title("Held-Out AUPRC with 95% Paired-Bootstrap Intervals")
plt.tight_layout()
plt.savefig(figure_4_2_path, dpi=FIGURE_DPI)
plt.close()

latency_plot = model_comparison.sort_values("p95_latency_ms", ascending=True).reset_index(drop=True)
latency_values = numeric_array(latency_plot, "p95_latency_ms", MODEL_COMPARISON_IN.name)
latency_labels = latency_plot["model_label"].astype("string").tolist()

figure_4_3_path = FIGURES_DIR / "figure_4_3_model_p95_latency.png"
plt.figure(figsize=(9.0, 5.4))
plt.barh(latency_labels, latency_values)
plt.xlabel("p95 inference latency (milliseconds)")
plt.title("Frozen Model Inference Latency")
for position, value in enumerate(latency_values.tolist()):
    plt.text(value, position, f" {value:.3f}", va="center")
plt.tight_layout()
plt.savefig(figure_4_3_path, dpi=FIGURE_DPI)
plt.close()

## Tables 4.4–4.5 and Figure 4.4 — Paired comparisons and hypothesis decisions

In [ ]:
TABLE_4_4_CAPTION = (
    "Paired held-out AUPRC differences. Positive differences favor the left-hand model "
    "named in each comparison."
)

table_4_4_rows: list[dict[str, object]] = []
for row_position in range(len(paired_differences)):
    row = paired_differences.iloc[row_position]
    table_4_4_rows.append(
        {
            "Comparison": require_text(row["comparison"], f"comparison row {row_position}"),
            "AUPRC difference": format_number(row["point_auprc_difference"], 4, signed=True),
            "95% CI": (
                f"[{format_number(row['bootstrap_ci_lower_95'], 4, signed=True)}, "
                f"{format_number(row['bootstrap_ci_upper_95'], 4, signed=True)}]"
            ),
            "Required margin": format_number(row["required_margin"], 4),
            "P(difference > 0)": format_percent(row["probability_difference_gt_zero"], 1),
            "P(meets required margin)": format_percent(
                row["probability_meets_required_margin"], 1
            ),
            "Point-margin result": "Pass" if require_bool(
                row["point_margin_pass"], "point margin pass"
            ) else "Fail",
        }
    )

table_4_4 = pd.DataFrame(table_4_4_rows)
table_4_4_paths = write_table_bundle(
    table_4_4,
    "table_4_4_paired_auprc_differences_v2",
    TABLE_4_4_CAPTION,
    "tab:paired-auprc-differences",
)
display(table_4_4)

TABLE_4_5_CAPTION = "Consolidated H1–H3 point criteria, uncertainty intervals, and decisions."
table_4_5_rows: list[dict[str, object]] = []
for row_position in range(len(hypothesis_evaluation)):
    row = hypothesis_evaluation.iloc[row_position]
    latency_value = optional_float(row["latency_value_ms"])
    latency_limit = optional_float(row["latency_limit_ms"])
    latency_text = "—"
    if latency_value is not None and latency_limit is not None:
        latency_text = f"{latency_value:.3f} ms (limit {latency_limit:.1f} ms)"
    table_4_5_rows.append(
        {
            "Hypothesis": require_text(row["hypothesis"], f"hypothesis row {row_position}"),
            "Research question": require_text(
                row["research_question"], f"research question row {row_position}"
            ),
            "Criterion": require_text(row["criterion"], f"criterion row {row_position}"),
            "Observed": format_number(row["observed_value"], 4, signed=True),
            "Required": format_number(row["required_value"], 4),
            "95% CI": (
                f"[{format_number(row['bootstrap_ci_lower_95'], 4, signed=True)}, "
                f"{format_number(row['bootstrap_ci_upper_95'], 4, signed=True)}]"
            ),
            "Latency": latency_text,
            "Decision": support_label(row["supported"]),
        }
    )

table_4_5 = pd.DataFrame(table_4_5_rows)
table_4_5_paths = write_table_bundle(
    table_4_5,
    "table_4_5_hypothesis_evaluation_v2",
    TABLE_4_5_CAPTION,
    "tab:hypothesis-evaluation",
)
display(table_4_5)

pair_plot = paired_differences.sort_values("point_auprc_difference", ascending=True).reset_index(drop=True)
pair_points = numeric_array(pair_plot, "point_auprc_difference", PAIRED_DIFFERENCES_IN.name)
pair_lower = numeric_array(pair_plot, "bootstrap_ci_lower_95", PAIRED_DIFFERENCES_IN.name)
pair_upper = numeric_array(pair_plot, "bootstrap_ci_upper_95", PAIRED_DIFFERENCES_IN.name)
pair_labels = pair_plot["comparison"].astype("string").tolist()
pair_positions = np.arange(len(pair_plot), dtype=float)

figure_4_4_path = FIGURES_DIR / "figure_4_4_paired_auprc_differences.png"
plt.figure(figsize=(10.5, 5.5))
plt.errorbar(
    pair_points,
    pair_positions,
    xerr=np.vstack([pair_points - pair_lower, pair_upper - pair_points]),
    fmt="o",
    capsize=4,
)
plt.axvline(0.0, linewidth=1)
plt.yticks(pair_positions, pair_labels)
plt.xlabel("Paired held-out AUPRC difference")
plt.title("Paired Model Differences with 95% Bootstrap Intervals")
plt.tight_layout()
plt.savefig(figure_4_4_path, dpi=FIGURE_DPI)
plt.close()

## Table 4.6 and Figure 4.5 — Alignment-window sensitivity

In [ ]:
TABLE_4_6_CAPTION = (
    "Sensitivity of controlled flow coverage and binary targets to the temporal window. "
    "Models were not retrained for these descriptive window comparisons."
)

window_display = window_sensitivity.sort_values("window_half_width_s").reset_index(drop=True)
table_4_6_rows: list[dict[str, object]] = []
for row_position in range(len(window_display)):
    row = window_display.iloc[row_position]
    table_4_6_rows.append(
        {
            "Window": f"±{format_count(row['window_half_width_s'])} s",
            "Events": format_count(row["events"]),
            "Events with flows": format_count(row["events_with_flows"]),
            "Flow coverage": format_percent(row["flow_coverage_fraction"], 2),
            "Positive targets": format_count(row["positive_targets"]),
            "Target prevalence": format_percent(row["target_prevalence"], 2),
            "Target changes vs. primary": format_count(row["target_changes_vs_primary"]),
            "Positive-set Jaccard": format_number(row["positive_jaccard_vs_primary"], 4),
            "Boundary-truncated events": format_count(row["boundary_truncated_events"]),
        }
    )

table_4_6 = pd.DataFrame(table_4_6_rows)
table_4_6_paths = write_table_bundle(
    table_4_6,
    "table_4_6_alignment_window_sensitivity_v2",
    TABLE_4_6_CAPTION,
    "tab:alignment-window-sensitivity",
)
display(table_4_6)

window_values = numeric_array(window_display, "window_half_width_s", WINDOW_SENSITIVITY_IN.name)
coverage_values = numeric_array(
    window_display, "flow_coverage_fraction", WINDOW_SENSITIVITY_IN.name
)
prevalence_values = numeric_array(
    window_display, "target_prevalence", WINDOW_SENSITIVITY_IN.name
)

figure_4_5_path = FIGURES_DIR / "figure_4_5_alignment_window_sensitivity.png"
plt.figure(figsize=(7.8, 5.0))
plt.plot(window_values, coverage_values, marker="o", label="Flow coverage")
plt.plot(window_values, prevalence_values, marker="o", label="Target prevalence")
plt.xlabel("Temporal half-window (seconds)")
plt.ylabel("Fraction of candidate point events")
plt.ylim(0.0, 1.02)
plt.title("Alignment-Window Sensitivity")
plt.legend()
plt.tight_layout()
plt.savefig(figure_4_5_path, dpi=FIGURE_DPI)
plt.close()

## Table 4.7 and Figure 4.6 — Descriptive subgroup robustness

In [ ]:
TABLE_4_7_CAPTION = (
    "Descriptive range of subgroup AUPRC values under each model's frozen threshold. "
    "These summaries are diagnostic and are not additional hypothesis tests."
)

subgroup_summary_rows: list[dict[str, object]] = []
model_keys = sorted(subgroup_performance["model_key"].astype("string").unique().tolist())
for model_key_value in model_keys:
    model_key = require_text(model_key_value, "subgroup model key")
    model_frame = subgroup_performance.loc[
        subgroup_performance["model_key"].astype("string").eq(model_key)
    ].copy()
    if model_frame.empty:
        continue
    model_frame["auprc"] = pd.to_numeric(model_frame["auprc"], errors="raise")
    model_frame = model_frame.sort_values("auprc").reset_index(drop=True)
    worst_row = model_frame.iloc[0]
    best_row = model_frame.iloc[-1]
    auprc_values = numeric_array(model_frame, "auprc", SUBGROUP_IN.name)
    subgroup_summary_rows.append(
        {
            "Model": require_text(model_frame.iloc[0]["model_label"], "subgroup model label"),
            "Reported subgroups": f"{len(model_frame):,}",
            "Minimum subgroup AUPRC": f"{float(np.min(auprc_values)):.4f}",
            "Median subgroup AUPRC": f"{float(np.median(auprc_values)):.4f}",
            "Maximum subgroup AUPRC": f"{float(np.max(auprc_values)):.4f}",
            "Lowest subgroup": (
                f"{require_text(worst_row['subgroup_variable'], 'worst subgroup variable')}: "
                f"{require_text(worst_row['subgroup_value'], 'worst subgroup value')}"
            ),
            "Highest subgroup": (
                f"{require_text(best_row['subgroup_variable'], 'best subgroup variable')}: "
                f"{require_text(best_row['subgroup_value'], 'best subgroup value')}"
            ),
        }
    )

table_4_7 = pd.DataFrame(subgroup_summary_rows)
table_4_7_paths = write_table_bundle(
    table_4_7,
    "table_4_7_subgroup_robustness_summary_v2",
    TABLE_4_7_CAPTION,
    "tab:subgroup-robustness",
)
display(table_4_7)

subgroup_plot_rows: list[dict[str, object]] = []
for model_key_value in model_keys:
    model_key = require_text(model_key_value, "subgroup plot model key")
    model_frame = subgroup_performance.loc[
        subgroup_performance["model_key"].astype("string").eq(model_key)
    ]
    if model_frame.empty:
        continue
    values = numeric_array(model_frame, "auprc", SUBGROUP_IN.name)
    subgroup_plot_rows.append(
        {
            "model_label": require_text(model_frame.iloc[0]["model_label"], "subgroup plot label"),
            "minimum": float(np.min(values)),
            "median": float(np.median(values)),
            "maximum": float(np.max(values)),
        }
    )

subgroup_plot = pd.DataFrame(subgroup_plot_rows).sort_values("median").reset_index(drop=True)
subgroup_min = numeric_array(subgroup_plot, "minimum", "subgroup plot")
subgroup_median = numeric_array(subgroup_plot, "median", "subgroup plot")
subgroup_max = numeric_array(subgroup_plot, "maximum", "subgroup plot")
subgroup_labels = subgroup_plot["model_label"].astype("string").tolist()
subgroup_positions = np.arange(len(subgroup_plot), dtype=float)

figure_4_6_path = FIGURES_DIR / "figure_4_6_subgroup_auprc_ranges.png"
plt.figure(figsize=(9.0, 5.4))
for position, minimum, median, maximum in zip(
    subgroup_positions.tolist(),
    subgroup_min.tolist(),
    subgroup_median.tolist(),
    subgroup_max.tolist(),
):
    plt.plot([minimum, maximum], [position, position], linewidth=2)
    plt.plot(median, position, marker="o")
plt.yticks(subgroup_positions, subgroup_labels)
plt.xlabel("Subgroup AUPRC")
plt.xlim(0.0, 1.02)
plt.title("Descriptive Subgroup AUPRC Ranges")
plt.tight_layout()
plt.savefig(figure_4_6_path, dpi=FIGURE_DPI)
plt.close()

## Table 4.8 and Figures 4.7–4.9 — Explainability synthesis

Importance magnitudes are ranked **within** their originating explanation method. Transformer gradient-times-input values, R-GCN feature attributions, and relation-weight norms are not treated as directly comparable effect sizes.

In [ ]:
TABLE_4_8_CAPTION = (
    "Highest-ranked Transformer and R-GCN explanation components within each source "
    "and explanation method. Importance magnitudes are not compared across methods."
)

explanation_group_columns = ["model_family", "component_type", "variant"]
explanation_groups = (
    explainability[explanation_group_columns]
    .astype("string")
    .drop_duplicates()
    .sort_values(explanation_group_columns)
    .reset_index(drop=True)
)

explanation_table_rows: list[dict[str, object]] = []
for group_position in range(len(explanation_groups)):
    group_row = explanation_groups.iloc[group_position]
    family = require_text(group_row["model_family"], "explanation family")
    component_type = require_text(group_row["component_type"], "explanation component type")
    variant = require_text(group_row["variant"], "explanation variant")
    group_frame = explainability.loc[
        explainability["model_family"].astype("string").eq(family)
        & explainability["component_type"].astype("string").eq(component_type)
        & explainability["variant"].astype("string").eq(variant)
    ].copy()
    group_frame["importance"] = pd.to_numeric(group_frame["importance"], errors="raise")
    group_frame = group_frame.sort_values("importance", ascending=False).head(
        TOP_EXPLANATION_COMPONENTS
    )
    for local_rank, row_position in enumerate(group_frame.index.tolist(), start=1):
        row = group_frame.loc[row_position]
        explanation_table_rows.append(
            {
                "Model family": family.replace("_", " ").title(),
                "Variant": variant.replace("_", " "),
                "Component type": component_type.replace("_", " "),
                "Rank": local_rank,
                "Component": require_text(row["component"], "explanation component"),
                "Importance metric": require_text(
                    row["importance_metric"], "explanation importance metric"
                ),
                "Importance": format_number(row["importance"], 6),
            }
        )

explanation_table = pd.DataFrame(explanation_table_rows)
table_4_8_paths = write_table_bundle(
    explanation_table,
    "table_4_8_explainability_summary_v2",
    TABLE_4_8_CAPTION,
    "tab:explainability-summary",
)
display(explanation_table.head(20))


def save_explanation_bar(
    frame: pd.DataFrame,
    output_path: Path,
    title: str,
    x_label: str,
) -> None:
    if frame.empty:
        return
    working = frame.copy()
    working["importance"] = pd.to_numeric(working["importance"], errors="raise")
    working = working.sort_values("importance", ascending=False).head(
        TOP_EXPLANATION_COMPONENTS
    ).sort_values("importance")
    labels = working["component"].astype("string").tolist()
    values = numeric_array(working, "importance", "explanation figure")
    plt.figure(figsize=(9.5, 5.8))
    plt.barh(labels, values)
    plt.xlabel(x_label)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_path, dpi=FIGURE_DPI)
    plt.close()


transformer_features = explainability.loc[
    explainability["model_family"].astype("string").eq("temporal_transformer")
    & explainability["component_type"].astype("string").eq("transformed_feature")
]
rgcn_event_features = explainability.loc[
    explainability["model_family"].astype("string").eq("rgcn")
    & explainability["component_type"].astype("string").eq("event_node_feature")
    & explainability["variant"].astype("string").eq("full_lag_relation")
]
rgcn_relations = explainability.loc[
    explainability["model_family"].astype("string").eq("rgcn")
    & explainability["component_type"].astype("string").str.contains("relation", na=False)
    & explainability["variant"].astype("string").eq("full_lag_relation")
]

figure_4_7_path = FIGURES_DIR / "figure_4_7_transformer_feature_attribution.png"
figure_4_8_path = FIGURES_DIR / "figure_4_8_rgcn_event_feature_attribution.png"
figure_4_9_path = FIGURES_DIR / "figure_4_9_rgcn_relation_diagnostics.png"

save_explanation_bar(
    transformer_features,
    figure_4_7_path,
    "Temporal Transformer: Highest-Ranked Feature Attributions",
    "Mean absolute gradient × input",
)
save_explanation_bar(
    rgcn_event_features,
    figure_4_8_path,
    "Full R-GCN: Highest-Ranked Event-Node Attributions",
    "Mean absolute gradient × input",
)
save_explanation_bar(
    rgcn_relations,
    figure_4_9_path,
    "Full R-GCN: Relation-Weight Diagnostics",
    "Relation linear-map Frobenius norm",
)

## Chapter 4 draft narrative

In [ ]:
best_model_row = model_comparison.sort_values("test_auprc", ascending=False).iloc[0]
best_model_label = require_text(best_model_row["model_label"], "best model label")
best_model_auprc = require_float(best_model_row["test_auprc"], "best model test AUPRC")

h2_record = row_where(hypothesis_evaluation, "hypothesis", "H2", HYPOTHESES_IN.name)
h3_record = row_where(hypothesis_evaluation, "hypothesis", "H3", HYPOTHESES_IN.name)

all_event_processing_mae = require_float(
    all_events_row["processing_time_mae_s"],
    "all-event processing-time MAE",
)
h1_reduction = require_float(h1_record["observed_value"], "H1 observed reduction")
h1_lower = require_float(h1_record["bootstrap_ci_lower_95"], "H1 lower interval")
h1_upper = require_float(h1_record["bootstrap_ci_upper_95"], "H1 upper interval")
h2_difference = require_float(h2_record["observed_value"], "H2 observed difference")
h2_lower = require_float(h2_record["bootstrap_ci_lower_95"], "H2 lower interval")
h2_upper = require_float(h2_record["bootstrap_ci_upper_95"], "H2 upper interval")
h3_difference = require_float(h3_record["observed_value"], "H3 observed difference")
h3_lower = require_float(h3_record["bootstrap_ci_lower_95"], "H3 lower interval")
h3_upper = require_float(h3_record["bootstrap_ci_upper_95"], "H3 upper interval")
h3_latency = require_float(h3_record["latency_value_ms"], "H3 latency")
h3_latency_limit = require_float(h3_record["latency_limit_ms"], "H3 latency limit")

coverage_min = float(np.min(numeric_array(window_display, "flow_coverage_fraction", WINDOW_SENSITIVITY_IN.name)))
coverage_max = float(np.max(numeric_array(window_display, "flow_coverage_fraction", WINDOW_SENSITIVITY_IN.name)))
prevalence_min = float(np.min(numeric_array(window_display, "target_prevalence", WINDOW_SENSITIVITY_IN.name)))
prevalence_max = float(np.max(numeric_array(window_display, "target_prevalence", WINDOW_SENSITIVITY_IN.name)))

def first_component(frame: pd.DataFrame, name: str) -> str:
    if frame.empty:
        return "no component available"
    ordered = frame.copy()
    ordered["importance"] = pd.to_numeric(ordered["importance"], errors="raise")
    row = ordered.sort_values("importance", ascending=False).iloc[0]
    return require_text(row["component"], name)


top_transformer_component = first_component(
    transformer_features,
    "top Transformer component",
)
top_rgcn_event_component = first_component(
    rgcn_event_features,
    "top R-GCN event component",
)
top_rgcn_relation_component = first_component(
    rgcn_relations,
    "top R-GCN relation component",
)

chapter4_narrative = f"""# Chapter 4 — Results

## 4.1 Evaluation design

The final evaluation used {candidate_events:,} candidate low-altitude point events and a common held-out set of {common_test_events:,} events, including {test_positives:,} positive targets ({test_prevalence:.2%}). Table 4.1 summarizes the frozen evaluation design. All model comparisons used identical held-out event identifiers, and the paired bootstrap preserved the binary target strata. The aircraft and NF-UNSW-NB15-v3 observations were joined through a controlled temporal replay; therefore, these results characterize the analytical pipeline rather than observed site-coincident aircraft/cyber incidents.

## 4.2 RQ1: Timestamp alignment fidelity

Table 4.2 and Figure 4.1 compare aircraft-reported event time with snapshot/processing time. Event-time MAE was 0 seconds by definition relative to the event timestamp, whereas processing-time MAE was {all_event_processing_mae:.3f} seconds. The resulting MAE reduction was {h1_reduction:.3f} seconds (95% bootstrap CI {h1_lower:.3f} to {h1_upper:.3f}). H1 was {support_label(h1_record['supported']).lower()} under the frozen requirement of at least 1.0 second. This finding evaluates timestamp fidelity within the source records; it is not evidence of real cross-domain clock synchronization.

## 4.3 RQ2: Comparative predictive performance

Table 4.3 reports validation and held-out performance, while Figure 4.2 displays held-out AUPRC with paired-bootstrap uncertainty. The strongest held-out point estimate was produced by **{best_model_label}** with AUPRC {best_model_auprc:.4f}. The full R-GCN minus temporal Transformer AUPRC difference was {h2_difference:+.4f} (95% CI {h2_lower:+.4f} to {h2_upper:+.4f}). H2 was {support_label(h2_record['supported']).lower()} under the required +0.03 AUPRC margin. Figure 4.3 reports each model's frozen p95 latency using the measurement definition retained from its source notebook.

## 4.4 RQ3: Lag/relation contribution and latency

Table 4.4 presents paired AUPRC differences, and Table 4.5 consolidates all hypothesis decisions. The full R-GCN minus lag/relation-ablation difference was {h3_difference:+.4f} (95% CI {h3_lower:+.4f} to {h3_upper:+.4f}). Full-model local p95 latency was {h3_latency:.3f} ms against the {h3_latency_limit:.1f} ms limit. Because H3 required both an AUPRC gain of at least 0.05 and latency within the limit, H3 was {support_label(h3_record['supported']).lower()}.

## 4.5 Robustness and sensitivity

Table 4.6 and Figure 4.5 summarize the ±120-, ±300-, and ±600-second alignment windows without retraining the models. Flow coverage ranged from {coverage_min:.2%} to {coverage_max:.2%}, while target prevalence ranged from {prevalence_min:.2%} to {prevalence_max:.2%}. These changes describe sensitivity of the controlled temporal mapping, not observed changes in operational attack incidence. Table 4.7 and Figure 4.6 summarize frozen-threshold subgroup diagnostics. The subgroup results are descriptive because the available group sizes and class counts were not designed as separate inferential tests.

## 4.6 Explainability results

Table 4.8 and Figures 4.7–4.9 summarize within-method explanation rankings. The highest-ranked Transformer feature was `{top_transformer_component}`. The highest-ranked full R-GCN event-node feature was `{top_rgcn_event_component}`, and the highest-ranked full R-GCN relation diagnostic was `{top_rgcn_relation_component}`. Gradient-times-input values and relation-weight norms have different meanings and scales; rankings were therefore interpreted within their originating explanation method rather than as cross-model effect sizes.

## 4.7 Summary of hypothesis outcomes

H1 was {support_label(h1_record['supported']).lower()}, H2 was {support_label(h2_record['supported']).lower()}, and H3 was {support_label(h3_record['supported']).lower()}. The results support conclusions about timestamp fidelity and model behavior under the controlled multimodal alignment design. They do not demonstrate that an NF3 flow caused, accompanied, or was produced by a specific aircraft observation near PANC.
"""

NARRATIVE_OUT.write_text(chapter4_narrative, encoding="utf-8")
print(chapter4_narrative)

## Caption manifest, HTML report, provenance, and export package

In [ ]:
caption_rows = [
    {
        "artifact_type": "table",
        "number": "Table 4.1",
        "primary_file": "tables/table_4_1_evaluation_design_v2.csv",
        "caption": TABLE_4_1_CAPTION,
        "interpretive_note": "Controlled cross-domain alignment design; no causal or site-coincident claim.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.2",
        "primary_file": "tables/table_4_2_timestamp_alignment_v2.csv",
        "caption": TABLE_4_2_CAPTION,
        "interpretive_note": "Event-time error is zero by definition relative to the aircraft event timestamp.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.3",
        "primary_file": "tables/table_4_3_model_performance_v2.csv",
        "caption": TABLE_4_3_CAPTION,
        "interpretive_note": "Thresholds and latency definitions remain frozen from source notebooks.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.4",
        "primary_file": "tables/table_4_4_paired_auprc_differences_v2.csv",
        "caption": TABLE_4_4_CAPTION,
        "interpretive_note": "Positive values favor the left-hand model in the comparison label.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.5",
        "primary_file": "tables/table_4_5_hypothesis_evaluation_v2.csv",
        "caption": TABLE_4_5_CAPTION,
        "interpretive_note": "Point criteria determine support; intervals are uncertainty diagnostics.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.6",
        "primary_file": "tables/table_4_6_alignment_window_sensitivity_v2.csv",
        "caption": TABLE_4_6_CAPTION,
        "interpretive_note": "Window sensitivity was evaluated without model retraining.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.7",
        "primary_file": "tables/table_4_7_subgroup_robustness_summary_v2.csv",
        "caption": TABLE_4_7_CAPTION,
        "interpretive_note": "Subgroup results are descriptive, not separate hypothesis tests.",
    },
    {
        "artifact_type": "table",
        "number": "Table 4.8",
        "primary_file": "tables/table_4_8_explainability_summary_v2.csv",
        "caption": TABLE_4_8_CAPTION,
        "interpretive_note": "Importance magnitudes are not comparable across explanation methods.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.1",
        "primary_file": "figures/figure_4_1_timestamp_alignment_mae.png",
        "caption": "Mean absolute timestamp error for event-time and processing-time timestamps.",
        "interpretive_note": "Event-time MAE is zero by construction against the aircraft event timestamp.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.2",
        "primary_file": "figures/figure_4_2_model_test_auprc_ci.png",
        "caption": "Held-out AUPRC point estimates with 95% paired-bootstrap intervals.",
        "interpretive_note": "All models use the same common held-out events.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.3",
        "primary_file": "figures/figure_4_3_model_p95_latency.png",
        "caption": "Frozen p95 model-inference latency measurements.",
        "interpretive_note": "Measurement definitions differ by model family and are retained verbatim.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.4",
        "primary_file": "figures/figure_4_4_paired_auprc_differences.png",
        "caption": "Paired held-out AUPRC differences with 95% bootstrap intervals.",
        "interpretive_note": "Positive values favor the left-hand model in each comparison.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.5",
        "primary_file": "figures/figure_4_5_alignment_window_sensitivity.png",
        "caption": "Flow coverage and target prevalence across temporal alignment windows.",
        "interpretive_note": "Descriptive sensitivity analysis without retraining.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.6",
        "primary_file": "figures/figure_4_6_subgroup_auprc_ranges.png",
        "caption": "Minimum, median, and maximum descriptive subgroup AUPRC by model.",
        "interpretive_note": "Subgroup estimates are not separate inferential tests.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.7",
        "primary_file": "figures/figure_4_7_transformer_feature_attribution.png",
        "caption": "Highest-ranked temporal Transformer feature attributions.",
        "interpretive_note": "Gradient-times-input magnitudes are interpreted within the Transformer only.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.8",
        "primary_file": "figures/figure_4_8_rgcn_event_feature_attribution.png",
        "caption": "Highest-ranked full R-GCN event-node feature attributions.",
        "interpretive_note": "Gradient-times-input magnitudes are interpreted within this R-GCN source only.",
    },
    {
        "artifact_type": "figure",
        "number": "Figure 4.9",
        "primary_file": "figures/figure_4_9_rgcn_relation_diagnostics.png",
        "caption": "Full R-GCN relation-weight norm diagnostics.",
        "interpretive_note": "Weight norms are diagnostics, not causal relation effects.",
    },
]
caption_manifest = pd.DataFrame(caption_rows)
caption_manifest.to_csv(CAPTION_MANIFEST_OUT, index=False)

report_sections: list[str] = []
report_sections.append("<!doctype html><html><head><meta charset='utf-8'>")
report_sections.append("<title>Chapter 4 Results Exports</title>")
report_sections.append(
    "<style>body{font-family:Arial,sans-serif;max-width:1100px;margin:2rem auto;line-height:1.5;}"
    "table{border-collapse:collapse;width:100%;margin:1rem 0 2rem;}"
    "th,td{border:1px solid #bbb;padding:.45rem;text-align:left;}"
    "img{max-width:100%;height:auto;margin:1rem 0 2rem;}"
    "code{background:#f2f2f2;padding:.1rem .25rem;}</style></head><body>"
)
report_sections.append("<h1>Chapter 4 Results Exports</h1>")
report_sections.append(
    "<p><strong>Interpretation boundary:</strong> "
    + html.escape(interpretation_statement)
    + "</p>"
)

for table_number, table_frame, caption in [
    ("Table 4.1", table_4_1, TABLE_4_1_CAPTION),
    ("Table 4.2", table_4_2, TABLE_4_2_CAPTION),
    ("Table 4.3", table_4_3, TABLE_4_3_CAPTION),
    ("Table 4.4", table_4_4, TABLE_4_4_CAPTION),
    ("Table 4.5", table_4_5, TABLE_4_5_CAPTION),
    ("Table 4.6", table_4_6, TABLE_4_6_CAPTION),
    ("Table 4.7", table_4_7, TABLE_4_7_CAPTION),
    ("Table 4.8", explanation_table, TABLE_4_8_CAPTION),
]:
    report_sections.append(f"<h2>{html.escape(table_number)}</h2>")
    report_sections.append(f"<p>{html.escape(caption)}</p>")
    report_sections.append(table_frame.to_html(index=False, border=0))

for figure_number, figure_path in [
    ("Figure 4.1", figure_4_1_path),
    ("Figure 4.2", figure_4_2_path),
    ("Figure 4.3", figure_4_3_path),
    ("Figure 4.4", figure_4_4_path),
    ("Figure 4.5", figure_4_5_path),
    ("Figure 4.6", figure_4_6_path),
    ("Figure 4.7", figure_4_7_path),
    ("Figure 4.8", figure_4_8_path),
    ("Figure 4.9", figure_4_9_path),
]:
    if figure_path.is_file():
        relative_path = figure_path.relative_to(OUTPUT_DIR).as_posix()
        report_sections.append(f"<h2>{html.escape(figure_number)}</h2>")
        report_sections.append(
            f"<img src='{html.escape(relative_path)}' alt='{html.escape(figure_number)}'>"
        )

report_sections.append("</body></html>")
HTML_REPORT_OUT.write_text("\n".join(report_sections), encoding="utf-8")

configuration_payload = {
    "export_version": EXPORT_VERSION,
    "source_results_version": SOURCE_RESULTS_VERSION,
    "top_explanation_components": TOP_EXPLANATION_COMPONENTS,
    "figure_dpi": FIGURE_DPI,
    "table_formats": ["csv", "markdown", "latex", "html"],
    "interpretation": interpretation_statement,
}
CONFIG_OUT.write_text(json.dumps(configuration_payload, indent=2), encoding="utf-8")

reference_payload = {
    "export_version": EXPORT_VERSION,
    "source_results_version": SOURCE_RESULTS_VERSION,
    "source_results_sha256": sha256_file(REFERENCE_IN),
    "hypotheses": json.loads(hypothesis_evaluation.to_json(orient="records")),
    "models": json.loads(model_comparison.to_json(orient="records")),
    "table_files": sorted(
        path.relative_to(OUTPUT_DIR).as_posix() for path in TABLES_DIR.glob("*") if path.is_file()
    ),
    "figure_files": sorted(
        path.relative_to(OUTPUT_DIR).as_posix() for path in FIGURES_DIR.glob("*.png")
    ),
    "interpretation": interpretation_statement,
}
REFERENCE_OUT.write_text(
    json.dumps(reference_payload, indent=2, default=str),
    encoding="utf-8",
)

artifact_descriptions = {
    "tables/": "Numbered Chapter 4 tables in CSV, Markdown, LaTeX, and HTML formats.",
    "figures/": "High-resolution Chapter 4 PNG figures generated from frozen Notebook 07 outputs.",
    CAPTION_MANIFEST_OUT.name: "Table and figure numbering, captions, and interpretive notes.",
    INTEGRITY_OUT.name: "Notebook 07 checksum and structural-integrity verification results.",
    NARRATIVE_OUT.name: "Automatically generated Chapter 4 results draft grounded in frozen outputs.",
    HTML_REPORT_OUT.name: "Self-contained index page referencing all exported tables and figures.",
    REFERENCE_OUT.name: "Machine-readable Chapter 4 export reference.",
    CONFIG_OUT.name: "Frozen Notebook 08 export configuration.",
    BUILD_SUMMARY_OUT.name: "Notebook 08 provenance, environment, and output inventory.",
    CHECKSUMS_OUT.name: "SHA-256 manifest for the Chapter 4 export directory.",
    PACKAGE_OUT.name: "ZIP package containing the complete Chapter 4 export directory.",
}
pd.DataFrame(
    [
        {"artifact": artifact, "artifact_role": description}
        for artifact, description in artifact_descriptions.items()
    ]
).to_csv(DATA_DICTIONARY_OUT, index=False)

build_summary_payload = {
    "definitions": configuration_payload,
    "inputs": {
        "source_directory": RESULTS_DIR.relative_to(PROJECT_ROOT).as_posix(),
        "source_checksum_entries_verified": len(source_manifest),
        "source_integrity_checks_verified": len(source_integrity),
        "candidate_events": candidate_events,
        "common_test_events": common_test_events,
    },
    "outputs": {
        "table_bundle_count": 8,
        "table_file_count": len(list(TABLES_DIR.glob("*"))),
        "figure_file_count": len(list(FIGURES_DIR.glob("*.png"))),
        "caption_rows": len(caption_manifest),
        "narrative_file": NARRATIVE_OUT.name,
        "html_report_file": HTML_REPORT_OUT.name,
        "package_file": PACKAGE_OUT.name,
    },
    "interpretation": {
        "causal_claim": False,
        "site_coincident_claim": False,
        "statement": interpretation_statement,
    },
    "environment": {
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version,
        "platform": platform.platform(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    },
}
BUILD_SUMMARY_OUT.write_text(
    json.dumps(build_summary_payload, indent=2, default=str),
    encoding="utf-8",
)

checksum_candidates = sorted(
    path
    for path in OUTPUT_DIR.rglob("*")
    if path.is_file() and path != CHECKSUMS_OUT
)
checksum_lines = [
    f"{sha256_file(path)}  {path.relative_to(OUTPUT_DIR).as_posix()}"
    for path in checksum_candidates
]
CHECKSUMS_OUT.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")

if PACKAGE_OUT.exists():
    PACKAGE_OUT.unlink()
with zipfile.ZipFile(PACKAGE_OUT, mode="w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(BASE_DIR).as_posix())

print("Notebook 08 completed successfully.")
print("Tables:", len(list(TABLES_DIR.glob("*"))))
print("Figures:", len(list(FIGURES_DIR.glob("*.png"))))
print("Narrative:", NARRATIVE_OUT)
print("HTML report:", HTML_REPORT_OUT)
print("ZIP package:", PACKAGE_OUT)